# Stickler + Strands Evals: field-level structured-output evaluation

[strands-agents/evals#310](https://github.com/strands-agents/evals/issues/310) asks for a stickler integration; this notebook is the working proof.

**The gap.** Strands Evals ships deterministic evaluators for structured output, but the strongest is `Equals`: whole-object `==`, scoring 0.0 or 1.0. An extraction that gets 9 of 10 fields right scores identically to one that gets none right, and reordered list items count as wrong.

**The fix.** [Stickler](https://github.com/awslabs/stickler) compares structured objects field by field: type-aware comparators (dates as dates, amounts as numbers, free text fuzzily), order-independent list matching, and confusion-matrix metrics. Its zero-config `stickler.evaluate()` needs nothing but the Pydantic model the agent already uses.

**The integration.** One `Evaluator` subclass, ~30 lines, shown below. Everything else is stock Strands Evals: `Case`, `Experiment`, `EvaluationReport`.

> Runs fully offline: the "agent" is a stub returning realistic extractions, so no Bedrock credentials are needed. Swap in a real agent with one line (shown at the end).


In [ ]:
import datetime
from typing import List, Optional

import strands_evals
from pydantic import BaseModel

import stickler

print("stickler", stickler.__version__, "| strands-evals", strands_evals.__version__ if hasattr(strands_evals, "__version__") else "1.x")

## 1. The model your agent already uses

Nothing stickler-specific: this is the plain Pydantic model you would hand to
`agent(prompt, structured_output_model=Invoice)`.

In [ ]:
class LineItem(BaseModel):
    sku: str
    description: str
    quantity: int
    unit_price: float


class Invoice(BaseModel):
    invoice_id: str
    vendor_name: str
    invoice_date: datetime.date
    total_amount: float
    notes: Optional[str] = None
    line_items: List[LineItem] = []

## 2. The integration: one Evaluator subclass

The whole adapter is [examples/integrations/strands_evals/stickler_evaluator.py](../integrations/strands_evals/stickler_evaluator.py),
written in the shape it would take inside `strands-agents/evals` as
`src/strands_evals/evaluators/stickler.py`. It subclasses their `Evaluator`,
reads their `EvaluationData`, and returns their `EvaluationOutput`:

| stickler | strands-evals `EvaluationOutput` |
|---|---|
| `overall_score` (weighted field average) | `score` |
| `matched` (every field at/above its threshold) | `test_pass` |
| lowest-scoring fields | `reason` |

`stickler.eval_for(Invoice)` compiles the field-level comparison plan once
(per-field comparators inferred from types and names); each `evaluate()` call
reuses it. Accepts model instances, dicts, or JSON strings as either side.

In [ ]:
import sys

# The evaluator lives in examples/integrations/strands_evals/ so the demo and
# the code proposed upstream are the same code, not two copies.
sys.path.insert(0, "../integrations/strands_evals")
from stickler_evaluator import StructuredOutputEvaluator  # noqa: E402

print(StructuredOutputEvaluator.__doc__.strip().split("\n")[0])

## 3. Cases and a stand-in agent

Three labeled documents. The stub returns what a real extraction agent
plausibly would: one perfect, one with the usual small variations
(abbreviated vendor, reworded notes, reordered list), one badly wrong.

In [ ]:
from strands_evals import Case, Experiment

GROUND_TRUTH = {
    "doc-perfect": Invoice(
        invoice_id="INV-2024-0042", vendor_name="Acme Corporation",
        invoice_date=datetime.date(2024, 3, 15), total_amount=1247.50,
        notes="Net 30 payment terms",
        line_items=[
            LineItem(sku="WM-100", description="Wireless Mouse", quantity=2, unit_price=29.99),
            LineItem(sku="UC-050", description="USB-C Cable 1m", quantity=5, unit_price=12.99),
        ],
    ),
    "doc-close": Invoice(
        invoice_id="INV-2024-0043", vendor_name="Globex Corporation",
        invoice_date=datetime.date(2024, 4, 1), total_amount=890.00,
        notes="Payment due in 45 days",
        line_items=[
            LineItem(sku="KB-220", description="Mechanical Keyboard", quantity=3, unit_price=89.00),
            LineItem(sku="MP-010", description="Mouse Pad XL", quantity=7, unit_price=8.99),
        ],
    ),
    "doc-wrong": Invoice(
        invoice_id="INV-2024-0044", vendor_name="Initech LLC",
        invoice_date=datetime.date(2024, 5, 20), total_amount=15300.00,
        notes="Prepaid",
        line_items=[LineItem(sku="SRV-01", description="Consulting", quantity=40, unit_price=382.50)],
    ),
}

AGENT_OUTPUTS = {
    "doc-perfect": GROUND_TRUTH["doc-perfect"].model_copy(deep=True),
    "doc-close": Invoice(
        invoice_id="INV-2024-0043",
        vendor_name="Globex Corp",                      # abbreviated
        invoice_date=datetime.date(2024, 4, 1),
        total_amount=890.00,
        notes="due in 45 days",                          # reworded
        line_items=[                                     # reordered
            LineItem(sku="MP-010", description="XL Mouse Pad", quantity=7, unit_price=8.99),
            LineItem(sku="KB-220", description="Mechanical Keyboard", quantity=3, unit_price=89.00),
        ],
    ),
    "doc-wrong": Invoice(
        invoice_id="INV-2024-0099",                      # wrong id
        vendor_name="Initrode",                          # wrong vendor
        invoice_date=datetime.date(2024, 5, 2),          # wrong date
        total_amount=1530.00,                            # off by 10x
        notes=None,
        line_items=[],                                   # missed entirely
    ),
}


def agent_task(case: Case) -> Invoice:
    # Real agent: return agent(case.input, structured_output_model=Invoice).structured_output
    return AGENT_OUTPUTS[case.name]


cases = [
    Case(name=name, input=f"Extract the invoice from document {name}", expected_output=gt)
    for name, gt in GROUND_TRUTH.items()
]

## 4. Run it: stock Strands Evals harness

In [ ]:
experiment = Experiment(cases=cases, evaluators=[StructuredOutputEvaluator(Invoice)])
report = experiment.run_evaluations(agent_task)

for case, score, passed, reason in zip(report.cases, report.scores, report.test_passes, report.reasons):
    print(f"{case['name']:14} score={score:.3f}  pass={str(passed):5}  {reason}")
print(f"\noverall_score: {report.overall_score:.3f}")

Contrast with what `Equals` (the current deterministic evaluator) reports on
the same three cases: 1.0, 0.0, 0.0. The `doc-close` extraction, which any
human would call a near-perfect job, is indistinguishable from `doc-wrong`.
Field-level scoring separates them, and `reason` names exactly which fields to
look at.

## 5. Every decision is auditable

Nobody configured a comparator above, but every choice is inspectable and
defensible (a requirement for eval numbers that end up in front of
stakeholders):

In [ ]:
spec = stickler.eval_for(Invoice)
for field, info in spec.explain().items():
    print(f"{field:24} {info['comparator']:40} threshold={info['threshold']}  src={info['source']}")

And for a specific pair, `result.explain()` adds what actually happened,
distinguishing a near-miss from a total mismatch:

In [ ]:
result = spec.evaluate(GROUND_TRUTH["doc-close"], AGENT_OUTPUTS["doc-close"])
entry = result.explain()["vendor_name"]
print({k: entry[k] for k in ("score", "raw_similarity", "verdict") if k in entry})

## 6. Graduating beyond zero-config

When a team disagrees with an inferred decision, the same inference is
available as a regular `StructuredModel` constructor. Export its config, edit
any comparator, threshold, or weight, and rebuild: every existing stickler
tool (bulk evaluation, HTML reports) takes it from there:

In [ ]:
from stickler import StructuredModel

InvoiceEval = StructuredModel.from_pydantic(Invoice)
config = InvoiceEval.to_stickler_config()
config["fields"]["vendor_name"]["threshold"] = 0.6   # e.g. relax the name threshold
Tuned = StructuredModel.model_from_json(config)

# Rebuilt models take the JSON wire form (ISO strings for dates):
gt = Tuned.from_json(GROUND_TRUTH["doc-close"].model_dump(mode="json"))
pred = Tuned.from_json(AGENT_OUTPUTS["doc-close"].model_dump(mode="json"))
print("tuned vendor_name score:", gt.compare_with(pred)["field_scores"]["vendor_name"])

## Proposed upstream shape

Per [evals#310](https://github.com/strands-agents/evals/issues/310) and
[stickler discussion #164](https://github.com/awslabs/stickler/discussions/164):

- `StructuredOutputEvaluator` lands in `strands_evals/evaluators/`, gated
  behind an optional extra: `pip install "strands-agents-evals[stickler]"`
  (`stickler = ["stickler-eval>=0.5.0"]`), matching the existing
  `langfuse`/`langchain` extras pattern.
- Deterministic-only (no LLM comparators), extending the direction of the
  deterministic-evaluators epic (evals#109): cost-free, reproducible, no
  credentials.
- To run against a live agent, replace the stub:
  `return agent(case.input, structured_output_model=Invoice).structured_output`
